In [5]:
# ==============================================================================
# Notebook 05: Continuous Integration & Deployment (CI/CD) Pipeline
# ==============================================================================
import boto3
import time

print("🚀 Initiating Real AWS CI/CD Pipeline...")

# --- 1. Pipeline Setup ---
sts = boto3.client('sts')
region = boto3.Session().region_name
aws_account = sts.get_caller_identity()['Account']

role_arn = sts.get_caller_identity()['Arn']
if "assumed-role" in role_arn:
    role_name = role_arn.split('/')[1]
    role = f"arn:aws:iam::{aws_account}:role/{role_name}"
else:
    role = role_arn

S3_BUCKET = f"sagemaker-{region}-{aws_account}"
PREFIX_XGB = "quantamental-platform/xgboost-data"
DATA_CAPTURE_S3 = f"s3://{S3_BUCKET}/{PREFIX_XGB}/data-capture/"

sm_client = boto3.client('sagemaker')
endpoint_name = "quantamental-xgboost-live-v1" 

# --- Dynamically Extract the Correct XGBoost Container ---
print("Locating the correct XGBoost container image from your previous successful run...")
completed_jobs = sm_client.list_training_jobs(
    StatusEquals='Completed', 
    SortBy='CreationTime', 
    SortOrder='Descending'
)['TrainingJobSummaries']

if not completed_jobs:
    raise Exception("No completed training jobs found to extract the image URI from.")

previous_job_name = completed_jobs[0]['TrainingJobName']
xgboost_image = sm_client.describe_training_job(TrainingJobName=previous_job_name)['AlgorithmSpecification']['TrainingImage']
print(f"✅ Secure Image Found: {xgboost_image}")


# --- 2. Automated Training (The "CI" Phase) ---
new_job_name = f"cicd-xgb-train-{int(time.time())}"
print(f"\n[Step 1] Triggering Automated Retraining: {new_job_name}")

sm_client.create_training_job(
    TrainingJobName=new_job_name,
    RoleArn=role,
    AlgorithmSpecification={'TrainingImage': xgboost_image, 'TrainingInputMode': 'File'},
    ResourceConfig={'InstanceType': 'ml.m5.xlarge', 'InstanceCount': 1, 'VolumeSizeInGB': 5},
    StoppingCondition={'MaxRuntimeInSeconds': 3600},
    HyperParameters={"objective": "reg:squarederror", "num_round": "100", "max_depth": "5", "eta": "0.1", "subsample": "0.8"},
    InputDataConfig=[
        {
            'ChannelName': 'train', 
            'DataSource': {'S3DataSource': {'S3DataType': 'S3Prefix', 'S3Uri': f"s3://{S3_BUCKET}/{PREFIX_XGB}/train/", 'S3DataDistributionType': 'FullyReplicated'}}, 
            'ContentType': 'text/csv'
        },
        {
            'ChannelName': 'validation', 
            'DataSource': {'S3DataSource': {'S3DataType': 'S3Prefix', 'S3Uri': f"s3://{S3_BUCKET}/{PREFIX_XGB}/validation/", 'S3DataDistributionType': 'FullyReplicated'}}, 
            'ContentType': 'text/csv'
        }
    ],
    OutputDataConfig={'S3OutputPath': f"s3://{S3_BUCKET}/{PREFIX_XGB}/output"}
)

print("Waiting for AWS training cluster to finish compiling the new model (~3-4 mins)...")
while True:
    status = sm_client.describe_training_job(TrainingJobName=new_job_name)['TrainingJobStatus']
    if status == 'Completed':
        print("✅ Continuous Integration complete. New model generated.")
        break
    elif status == 'Failed':
        raise Exception("CI Pipeline Failed during training. Please run the diagnostic script again.")
    time.sleep(30)


# --- 3. Automated Deployment Prep (The "CD" Phase) ---
print("\n[Step 2] Initiating Continuous Deployment Preparation...")

job_details = sm_client.describe_training_job(TrainingJobName=new_job_name)
new_model_s3_uri = job_details['ModelArtifacts']['S3ModelArtifacts']
new_model_name = f"cicd-quantamental-model-{int(time.time())}"

print(f"Registering new model in AWS: {new_model_name}")
sm_client.create_model(
    ModelName=new_model_name, 
    PrimaryContainer={'Image': xgboost_image, 'ModelDataUrl': new_model_s3_uri}, 
    ExecutionRoleArn=role
)

new_config_name = f"cicd-config-{int(time.time())}"
print(f"Building new endpoint configuration: {new_config_name}")
sm_client.create_endpoint_config(
    EndpointConfigName=new_config_name,
    ProductionVariants=[{
        'VariantName': 'AllTraffic', 'ModelName': new_model_name, 
        'InitialInstanceCount': 1, 'InstanceType': 'ml.m5.large', 'InitialVariantWeight': 1.0
    }],
    DataCaptureConfig={
        'EnableCapture': True, 'InitialSamplingPercentage': 100,
        'DestinationS3Uri': DATA_CAPTURE_S3,
        'CaptureOptions': [{'CaptureMode': 'Input'}, {'CaptureMode': 'Output'}],
        'CaptureContentTypeHeader': {'CsvContentTypes': ['text/csv']}
    }
)

# --- 4. The Hot-Swap Attempt ---
print(f"\n[Step 3] Attempting to hot-swap live endpoint '{endpoint_name}'...")
try:
    sm_client.update_endpoint(EndpointName=endpoint_name, EndpointConfigName=new_config_name)
    print("Zero-downtime deployment initiated! AWS is swapping the hardware...")
    
    while True:
        status = sm_client.describe_endpoint(EndpointName=endpoint_name)['EndpointStatus']
        if status == 'InService':
            print("✅ Continuous Deployment complete! Live traffic is now flowing to the updated model.")
            break
        elif status == 'Failed':
            raise Exception("CD Pipeline Failed during endpoint update.")
        time.sleep(30)
        
except sm_client.exceptions.ClientError as e:
    if "Could not find endpoint" in str(e):
        print("✅ SUCCESS: The CI/CD pipeline built the new model and release configurations perfectly.")
        print("⚠️ NOTE: The live endpoint is currently shut down to save money, so the final hot-swap was safely bypassed.")
    else:
        print(f"Deployment Error: {e}")

print("\n🎉 CI/CD Pipeline Execution Finished!")

🚀 Initiating Real AWS CI/CD Pipeline...
Locating the correct XGBoost container image from your previous successful run...
✅ Secure Image Found: 683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:1.7-1

[Step 1] Triggering Automated Retraining: cicd-xgb-train-1781417969


Waiting for AWS training cluster to finish compiling the new model (~3-4 mins)...


✅ Continuous Integration complete. New model generated.

[Step 2] Initiating Continuous Deployment Preparation...
Registering new model in AWS: cicd-quantamental-model-1781418120


Building new endpoint configuration: cicd-config-1781418121



[Step 3] Attempting to hot-swap live endpoint 'quantamental-xgboost-live-v1'...


✅ SUCCESS: The CI/CD pipeline built the new model and release configurations perfectly.
⚠️ NOTE: The live endpoint is currently shut down to save money, so the final hot-swap was safely bypassed.

🎉 CI/CD Pipeline Execution Finished!
